# Persian Animated Video v2

این نسخه اصلاً `gTTS(lang=fa)` ندارد. صدای فارسی را با Edge TTS می‌سازد و در صورت نبود اینترنت، با espeak-ng فارسی fallback می‌کند.

In [ ]:
TOPIC = 'چرا آسمان آبی است؟' #@param {type:'string'}
EPISODE_ID = 'persian-v2-001' #@param {type:'string'}
VERTICAL = False #@param {type:'boolean'}

In [ ]:
import subprocess,sys,asyncio,math,random,shutil,zipfile,json,textwrap,wave,struct
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','edge-tts','Pillow','arabic-reshaper','python-bidi'],check=True)
subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core espeak-ng'],check=True)
from PIL import Image,ImageDraw,ImageFont
from bidi.algorithm import get_display
import arabic_reshaper
from google.colab import files
root=Path('/content')/EPISODE_ID; shutil.rmtree(root,ignore_errors=True); frames=root/'frames'; frames.mkdir(parents=True)
W,H=(720,1280) if VERTICAL else (1280,720)
script=[f'امروز درباره {TOPIC} صحبت می کنیم.',f'پشت این موضوع یک داستان علمی جذاب وجود دارد.', 'با دیدن سازوکار آن، پاسخ کاملاً روشن می شود.', f'حالا می دانیم که {TOPIC} فقط یک واقعیت ساده نیست.', 'برای داستان های علمی بیشتر، کانال را دنبال کنید.']
fa=lambda s:get_display(arabic_reshaper.reshape(s))
regular='/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf'; bold='/usr/share/fonts/truetype/noto/NotoNaskhArabic-Bold.ttf'
tts=' '.join(script); audio=root/'narration.mp3'; voice='fa-IR-DilaraNeural'
async def speak():
    import edge_tts
    await edge_tts.Communicate(tts,voice).save(str(audio))
try:
    asyncio.run(speak())
except Exception as err:
    print('Edge TTS unavailable:',err)
    audio=root/'narration.wav'; subprocess.run(['espeak-ng','-v','fa','-s','145','-w',str(audio),tts],check=False); voice='espeak-ng fa'
duration=max(20,len(tts)/8); fps=12; total=int(duration*fps); random.seed(7); particles=[(random.randrange(W),random.randrange(H),random.randrange(2,8),random.random()*6.28) for _ in range(50)]
for i in range(total):
    t=i/fps; scene=min(len(script)-1,int(t/(duration/len(script)))); img=Image.new('RGB',(W,H),(7,15,45)); d=ImageDraw.Draw(img)
    for y in range(H): d.line((0,y,W,y),fill=(int(8+22*y/H),int(20+45*y/H),int(62+80*y/H)))
    for x,y,r,p in particles:
        xx=int((x+45*math.sin(t*.7+p))%W); yy=int((y+35*math.cos(t*.5+p))%H); d.ellipse((xx-r,yy-r,xx+r,yy+r),fill=(70,170,240))
    font=ImageFont.truetype(bold,max(30,int(W*.045))); y0=int(H*.45)
    for line in textwrap.wrap(script[scene],width=24 if not VERTICAL else 14):
        s=fa(line); box=d.textbbox((0,0),s,font=font); x=(W-(box[2]-box[0]))//2; d.text((x+3,y0+3),s,font=font,fill='black'); d.text((x,y0),s,font=font,fill='white'); y0+=int(font.size*1.3)
    img.save(frames/f'frame-{i:05d}.jpg',quality=90)
video=root/'video.mp4'; cmd=['ffmpeg','-y','-nostdin','-loglevel','error','-framerate',str(fps),'-i',str(frames/'frame-%05d.jpg'),'-i',str(audio),'-c:v','libx264','-c:a','aac','-shortest','-pix_fmt','yuv420p',str(video)]; run=subprocess.run(cmd,capture_output=True,text=True)
if run.returncode!=0: print(run.stderr); raise RuntimeError('ساخت ویدیو شکست خورد')
(root/'script-fa.txt').write_text('\n\n'.join(script),encoding='utf-8'); (root/'manifest.json').write_text(json.dumps({'language':'fa','voice':voice,'animated':True,'video':'video.mp4'},ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path('/content')/f'{EPISODE_ID}.zip';
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(root.parent))
files.download(str(archive)); print('SUCCESS:',video,voice)